In [4]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

from src.data import load_table, to_binary, split_xy
from src.features import build_preprocessor
from src.models import get_models, fit_all
from src.evaluate import compare, cutoff_table


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
train = load_table("../data/KDDTrain.parquet")
test  = load_table("../data/KDDTest.parquet")

X_train, labels_train = split_xy(train, "class", extra_drop=["classnum"])
X_test,  labels_test  = split_xy(test,  "class", extra_drop=['classnum'])
y_train = to_binary(labels_train, "normal")
y_test  = to_binary(labels_test,  "normal")

preprocessor, cat_cols, num_cols = build_preprocessor(X_train)
print(len(cat_cols), "categorical,", len(num_cols), "numeric")


3 categorical, 33 numeric


In [26]:
fitted = fit_all(get_models(), preprocessor, X_train, y_train)
compare(fitted, X_test, y_test)

,model,accuracy,attack_precision,attack_recall,attack_f1,missed_attacks,false_alarms
0,Random Forest,0.790,0.969,0.652,0.779,4469,266
1,Logistic Regression,0.752,0.916,0.621,0.740,4863,734
2,XGBoost,0.794,0.968,0.660,0.785,4368,282


In [29]:
proba = fitted["Random Forest"].predict_proba(X_test)[:, 1]
cutoff_table("RF", y_test, proba)

,model,accuracy,attack_precision,attack_recall,attack_f1,missed_attacks,false_alarms
0,RF @ 0.5,0.792,0.969,0.655,0.782,4421,266
1,RF @ 0.2,0.851,0.924,0.805,0.860,2498,856
2,RF @ 0.1,0.894,0.923,0.887,0.905,1452,948
3,RF @ 0.05,0.911,0.920,0.924,0.922,969,1037
